# 🔢 Handwritten Digit Recognition — MNIST
### Neural Networks Course Project
**Model:** Multilayer Perceptron (MLP) | **Framework:** PyTorch

---
**Table of Contents**
1. Setup & Imports
2. Dataset Loading & Preprocessing
3. MLP Model Definition
4. Training & Evaluation Utilities
5. Experiment 1 — Baseline (ReLU, lr=0.001)
6. Experiment 2 — Different Activation (Sigmoid, lr=0.001)
7. Experiment 3 — Higher Learning Rate (ReLU, lr=0.01)
8. Results Comparison
9. Visualizations

## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import pandas as pd

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Device ───────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

## 2. Dataset Loading & Preprocessing

**Dataset:** MNIST — 70,000 grayscale images (28×28 px) of handwritten digits (0–9).
- 60,000 training samples → split into 50,000 train / 10,000 validation
- 10,000 test samples (held out until final evaluation)

**Preprocessing applied:**
- Each pixel value is **normalized** from [0, 255] → mean=0.1307, std=0.3081 (MNIST dataset statistics)
- Images are **flattened** from 28×28 to a 784-dimensional vector for the MLP input layer
- No missing values exist in MNIST; no categorical encoding is needed

In [ ]:
# ── Normalization transform (MNIST mean & std) ────────────────────────────────
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # Standard MNIST statistics
])

# ── Download datasets ─────────────────────────────────────────────────────────
full_train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset       = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# ── Train / Validation split (50k / 10k) ─────────────────────────────────────
train_size = 50000
val_size   = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(
    full_train_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

# ── DataLoaders ───────────────────────────────────────────────────────────────
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train samples     : {len(train_dataset):,}')
print(f'Validation samples: {len(val_dataset):,}')
print(f'Test samples      : {len(test_dataset):,}')

In [ ]:
# ── Visualize sample images ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
fig.suptitle('MNIST Dataset — Sample Images', fontsize=14, fontweight='bold')

for digit in range(10):
    # Find first occurrence of this digit in test set
    idx = next(i for i, (_, label) in enumerate(test_dataset) if label == digit)
    img, label = test_dataset[idx]
    axes[0, digit].imshow(img.squeeze(), cmap='gray')
    axes[0, digit].set_title(f'Digit {label}', fontsize=9)
    axes[0, digit].axis('off')
    # Second row — another random sample
    idx2 = next(i for i, (_, lbl) in enumerate(test_dataset) if lbl == digit and i != idx)
    img2, _ = test_dataset[idx2]
    axes[1, digit].imshow(img2.squeeze(), cmap='gray')
    axes[1, digit].axis('off')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. MLP Model Definition

**Architecture:**
```
Input  → [784]  (flattened 28×28 image)
Hidden → [512]  → Activation → Dropout(0.3)
Hidden → [256]  → Activation → Dropout(0.3)
Hidden → [128]  → Activation
Output → [10]   (digit classes 0–9)
```
- **Loss function:** CrossEntropyLoss (includes Softmax; standard for multi-class)
- **Optimizer:** Adam
- **Dropout:** 0.3 — applied after the first two hidden layers to reduce overfitting (optional enhancement)

In [ ]:
class MLP(nn.Module):
    """
    Multilayer Perceptron for MNIST digit classification.

    Args:
        activation (str): Activation function — 'relu', 'sigmoid', or 'tanh'.
        hidden_sizes (list): Number of neurons in each hidden layer.
        dropout_rate (float): Dropout probability (0 = disabled).
    """
    def __init__(self, activation='relu', hidden_sizes=[512, 256, 128], dropout_rate=0.3):
        super(MLP, self).__init__()

        # ── Activation function selection ─────────────────────────────────────
        activation_fns = {
            'relu':    nn.ReLU(),
            'sigmoid': nn.Sigmoid(),
            'tanh':    nn.Tanh()
        }
        assert activation in activation_fns, f"activation must be one of {list(activation_fns.keys())}"
        act_fn = activation_fns[activation]

        # ── Build layers dynamically ──────────────────────────────────────────
        layers = []
        in_features = 28 * 28  # 784 — flattened MNIST image

        for i, hidden_size in enumerate(hidden_sizes):
            layers.append(nn.Linear(in_features, hidden_size))
            layers.append(activation_fns[activation])  # fresh instance per layer
            if dropout_rate > 0 and i < len(hidden_sizes) - 1:
                layers.append(nn.Dropout(dropout_rate))
            in_features = hidden_size

        # Output layer — 10 classes (no softmax; CrossEntropyLoss includes it)
        layers.append(nn.Linear(in_features, 10))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten: (batch, 1, 28, 28) → (batch, 784)
        return self.network(x)


# ── Quick model summary ───────────────────────────────────────────────────────
demo_model = MLP(activation='relu')
print(demo_model)
total_params = sum(p.numel() for p in demo_model.parameters() if p.requires_grad)
print(f'\nTotal trainable parameters: {total_params:,}')

## 4. Training & Evaluation Utilities

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    """Run one training epoch. Returns (avg_loss, accuracy)."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        predicted = outputs.argmax(dim=1)
        correct  += (predicted == labels).sum().item()
        total    += labels.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    """Evaluate model on a DataLoader. Returns (avg_loss, accuracy)."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            predicted  = outputs.argmax(dim=1)
            correct   += (predicted == labels).sum().item()
            total     += labels.size(0)

    return total_loss / total, correct / total


def run_experiment(name, activation, lr, hidden_sizes, epochs, dropout_rate=0.3):
    """
    Train and evaluate one MLP experiment.

    Returns a dict with history and final test metrics.
    """
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"  Activation: {activation} | LR: {lr} | Hidden: {hidden_sizes}")
    print(f"{'='*60}")

    model     = MLP(activation=activation, hidden_sizes=hidden_sizes, dropout_rate=dropout_rate).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss,   val_acc   = evaluate(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:02d}/{epochs}  "
                  f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc*100:.2f}%  "
                  f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc*100:.2f}%")

    # ── Final test evaluation ─────────────────────────────────────────────────
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    print(f"\n  ▶ FINAL TEST  Loss: {test_loss:.4f}  Accuracy: {test_acc*100:.2f}%")

    return {
        'name':      name,
        'model':     model,
        'history':   history,
        'test_loss': test_loss,
        'test_acc':  test_acc,
        'params': {
            'activation':   activation,
            'lr':           lr,
            'hidden_sizes': hidden_sizes,
            'dropout_rate': dropout_rate,
        }
    }

## 5. Experiment 1 — Baseline (ReLU, lr=0.001)

**Configuration:**
- Activation: **ReLU** — the standard choice; avoids vanishing gradients
- Learning rate: **0.001** — Adam optimizer default
- Architecture: 784 → 512 → 256 → 128 → 10
- Dropout: 0.3 on first two hidden layers

In [ ]:
EPOCHS = 20

exp1 = run_experiment(
    name         = 'Experiment 1 — ReLU, lr=0.001 (Baseline)',
    activation   = 'relu',
    lr           = 0.001,
    hidden_sizes = [512, 256, 128],
    epochs       = EPOCHS
)

## 6. Experiment 2 — Different Activation Function (Sigmoid, lr=0.001)

**What changes:** Activation function → **Sigmoid**

**Why:** Sigmoid outputs values in (0,1), which can cause **vanishing gradients** in deep networks. This experiment tests whether that limitation is visible even in a 3-hidden-layer MLP compared to ReLU.

In [ ]:
exp2 = run_experiment(
    name         = 'Experiment 2 — Sigmoid, lr=0.001',
    activation   = 'sigmoid',
    lr           = 0.001,
    hidden_sizes = [512, 256, 128],
    epochs       = EPOCHS
)

## 7. Experiment 3 — Higher Learning Rate (ReLU, lr=0.01)

**What changes:** Learning rate → **0.01** (10× the baseline)

**Why:** A higher learning rate can speed up convergence but may cause instability or overshoot the loss minimum. This experiment tests the sensitivity of the baseline model to learning rate.

In [ ]:
exp3 = run_experiment(
    name         = 'Experiment 3 — ReLU, lr=0.01 (High LR)',
    activation   = 'relu',
    lr           = 0.01,
    hidden_sizes = [512, 256, 128],
    epochs       = EPOCHS
)

## 8. Experiment 4 — Smaller Network / Fewer Neurons (ReLU, lr=0.001)

**What changes:** Hidden layer sizes → **[128, 64, 32]** (≈4× fewer neurons)

**Why:** Reducing the network capacity tests whether the baseline is over-parameterized for MNIST, or whether the larger architecture genuinely contributes to accuracy. A smaller network also trains faster and is less prone to overfitting — but may underfit if capacity is too low.

In [ ]:
exp4 = run_experiment(
    name         = 'Experiment 4 — ReLU, lr=0.001 (Small Network)',
    activation   = 'relu',
    lr           = 0.001,
    hidden_sizes = [128, 64, 32],
    epochs       = EPOCHS
)

## 8. Results Comparison

In [ ]:
experiments = [exp1, exp2, exp3, exp4]

# ── Comparison table ──────────────────────────────────────────────────────────
rows = []
for e in experiments:
    p = e['params']
    rows.append({
        'Experiment':       e['name'].split('—')[0].strip(),
        'Activation':       p['activation'].capitalize(),
        'Learning Rate':    p['lr'],
        'Hidden Layers':    str(p['hidden_sizes']),
        'Test Accuracy (%)': f"{e['test_acc']*100:.2f}",
        'Final Test Loss':  f"{e['test_loss']:.4f}",
        'Best Val Acc (%)': f"{max(e['history']['val_acc'])*100:.2f}"
    })

df = pd.DataFrame(rows)
print('\n' + '='*80)
print('EXPERIMENT COMPARISON TABLE')
print('='*80)
print(df.to_string(index=False))
print('='*80)

## 9. Visualizations
### 9.1 Training & Validation Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('Training vs. Validation Loss', fontsize=16, fontweight='bold', y=1.02)

colors = [('#2563eb', '#93c5fd'), ('#16a34a', '#86efac'), ('#dc2626', '#fca5a5'), ('#7c3aed', '#c4b5fd')]

for ax, exp, (c_train, c_val) in zip(axes, experiments, colors):
    epochs_range = range(1, len(exp['history']['train_loss']) + 1)
    ax.plot(epochs_range, exp['history']['train_loss'], color=c_train, lw=2, label='Train Loss')
    ax.plot(epochs_range, exp['history']['val_loss'],   color=c_val,   lw=2, label='Val Loss', linestyle='--')
    ax.set_title(exp['name'].split('—')[1].strip() if '—' in exp['name'] else exp['name'], fontsize=11)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xlim(1, EPOCHS)

plt.tight_layout()
plt.savefig('loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.2 Training & Validation Accuracy Curves

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('Training vs. Validation Accuracy', fontsize=16, fontweight='bold', y=1.02)

for ax, exp, (c_train, c_val) in zip(axes, experiments, colors):
    epochs_range = range(1, len(exp['history']['train_acc']) + 1)
    train_acc_pct = [a * 100 for a in exp['history']['train_acc']]
    val_acc_pct   = [a * 100 for a in exp['history']['val_acc']]

    ax.plot(epochs_range, train_acc_pct, color=c_train, lw=2, label='Train Acc')
    ax.plot(epochs_range, val_acc_pct,   color=c_val,   lw=2, label='Val Acc',   linestyle='--')
    ax.set_title(exp['name'].split('—')[1].strip() if '—' in exp['name'] else exp['name'], fontsize=11)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy (%)')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xlim(1, EPOCHS)
    ax.set_ylim(0, 100)

plt.tight_layout()
plt.savefig('accuracy_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.3 Combined Accuracy Comparison (All Experiments)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
line_styles = ['-', '--', ':']
exp_colors  = ['#2563eb', '#16a34a', '#dc2626']
short_names = ['Exp1: ReLU lr=0.001', 'Exp2: Sigmoid lr=0.001', 'Exp3: ReLU lr=0.01']

for exp, ls, color, label in zip(experiments, line_styles, exp_colors, short_names):
    val_acc_pct = [a * 100 for a in exp['history']['val_acc']]
    ax.plot(range(1, len(val_acc_pct) + 1), val_acc_pct, lw=2.5, ls=ls, color=color, label=label)

ax.set_title('Validation Accuracy — All Experiments', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Accuracy (%)')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_xlim(1, EPOCHS)

plt.tight_layout()
plt.savefig('comparison_val_acc.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.4 Confusion Matrix — Best Experiment (Exp 1)

In [ ]:
best_model = exp1['model']
best_model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = best_model(images.to(device))
        preds   = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=range(10), yticklabels=range(10))
ax.set_title('Confusion Matrix — Experiment 1 (Best Model)', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label',      fontsize=12)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report:')
print(classification_report(all_labels, all_preds, target_names=[str(i) for i in range(10)]))

### 9.5 Sample Predictions

In [ ]:
best_model.eval()
images_sample, labels_sample = next(iter(test_loader))
with torch.no_grad():
    outputs = best_model(images_sample.to(device))
    preds   = outputs.argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 10, figsize=(18, 4))
fig.suptitle('Sample Predictions — Experiment 1', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    ax.imshow(images_sample[i].squeeze(), cmap='gray')
    correct = preds[i].item() == labels_sample[i].item()
    color   = 'green' if correct else 'red'
    ax.set_title(f'P:{preds[i].item()} T:{labels_sample[i].item()}',
                 fontsize=8, color=color, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('P = Predicted, T = True  |  Green = Correct, Red = Wrong')

### 9.6 Final Summary Table

In [ ]:
print('\n' + '='*70)
print('              FINAL RESULTS SUMMARY')
print('='*70)
print(f"{'Experiment':<15} {'Activation':<12} {'LR':<8} {'Test Acc':>10} {'Test Loss':>12}")
print('-'*70)
for e in experiments:
    p = e['params']
    label = e['name'].split('—')[0].strip()
    print(f"{label:<15} {p['activation']:<12} {str(p['lr']):<8} "
          f"{e['test_acc']*100:>9.2f}%  {e['test_loss']:>12.4f}")
print('='*70)

best = max(experiments, key=lambda e: e['test_acc'])
print(f"\n✅ Best experiment: {best['name']}")
print(f"   Test Accuracy: {best['test_acc']*100:.2f}%  |  Test Loss: {best['test_loss']:.4f}")